In [3]:
import random
import numpy as np
import torch
import pandas as pd
SEED = 39

def set_seed(seed=SEED):
    random.seed(seed)          # teacher forcing: random.random()
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed()

dataset = pd.read_table("data/dataset_small.tsv", quoting=3, names=["ru_id", "russian", "en_id", "english"])
dataset.head()

,ru_id,russian,en_id,english
0,243,Один раз в жизни я делаю хорошее дело... И оно...,3257,For once in my life I'm doing a good deed... A...
1,5409,Давайте что-нибудь попробуем!,1276,Let's try something.
2,5410,Мне пора идти спать.,1277,I have to go to sleep.
3,5411,Что ты делаешь?,16492,What are you doing?
4,5411,Что ты делаешь?,511884,What do you make?


In [4]:
from src.preprocessing import preprocess_parallel_data
data, _ = preprocess_parallel_data(dataset)

In [5]:
from sklearn.model_selection import train_test_split

train, temp = train_test_split(data, train_size = 0.8, random_state = SEED, shuffle = True)
valid, test = train_test_split(temp, train_size = 0.5, random_state = SEED, shuffle = True)
print(f"train: {train.shape}, valid: {valid.shape}, test: {test.shape}")

train: (659259, 4), valid: (82407, 4), test: (82408, 4)


In [ ]:
from pathlib import Path
import sentencepiece as spm

TOKENIZER_DIR = Path("artifacts/tokenizers")

ru_model_path = TOKENIZER_DIR / "ru_bpe.model"
en_model_path = TOKENIZER_DIR / "en_bpe.model"

assert ru_model_path.exists(), f"Не найден: {ru_model_path}"
assert en_model_path.exists(), f"Не найден: {en_model_path}"

ru_sp = spm.SentencePieceProcessor(
    model_file=str(ru_model_path)
)

en_sp = spm.SentencePieceProcessor(
    model_file=str(en_model_path)
)

RU_PAD_IDX = ru_sp.pad_id()
EN_PAD_IDX = en_sp.pad_id()

print("RU vocab:", ru_sp.get_piece_size())
print("EN vocab:", en_sp.get_piece_size())

In [7]:
def encode_source(text):
    return ru_sp.encode(text, out_type=int)


def encode_target(text):
    token_ids = en_sp.encode(text, out_type=int)

    return (
        [en_sp.bos_id()] +
        token_ids +
        [en_sp.eos_id()]
    )

In [8]:
for split in [train, valid, test]:
    split["ru_ids"] = split["russian"].apply(encode_source)
    split["en_ids"] = split["english"].apply(encode_target)

In [9]:
RU_PAD_IDX = ru_sp.pad_id()
EN_PAD_IDX = en_sp.pad_id()

RU_UNK_IDX = ru_sp.unk_id()
EN_UNK_IDX = en_sp.unk_id()

EN_SOS_IDX = en_sp.bos_id()
EN_EOS_IDX = en_sp.eos_id()

print(RU_PAD_IDX, EN_PAD_IDX, EN_SOS_IDX, EN_EOS_IDX)

0 0 2 3


In [10]:
row = train.iloc[0]

print(row["russian"])
print(row["ru_ids"])
print(ru_sp.decode(row["ru_ids"]))

print(row["english"])
print(row["en_ids"])
print(en_sp.decode(row["en_ids"][1:-1]))

Вам надо было сразу же мне позвонить.
[473, 321, 218, 2152, 410, 102, 1744, 7804]
Вам надо было сразу же мне позвонить.
You should've phoned me right away.
[2, 86, 271, 7807, 23, 4020, 67, 418, 660, 7799, 3]
You should've phoned me right away.


In [11]:
from torch.utils.data import DataLoader

from src.dataset import (
    TranslationDataset,
    TranslationCollator,
)
BATCH_SIZE = 64
generator = torch.Generator()
generator.manual_seed(SEED)
train_dataset = TranslationDataset(train)
val_dataset = TranslationDataset(valid)
test_dataset = TranslationDataset(test)

collator = TranslationCollator(
    ru_pad_idx=RU_PAD_IDX,
    en_pad_idx=EN_PAD_IDX
)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collator,
    num_workers=0,
    generator=generator
)

valid_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collator,
    num_workers=0
)

In [12]:
sources, source_lengths, targets = next(iter(train_loader))

print("Sources:", sources.shape)
print("Lengths:", source_lengths[:10])
print("Targets:", targets.shape)

Sources: torch.Size([64, 26])
Lengths: tensor([ 7, 11,  6,  8, 14,  4,  4,  7, 13,  6])
Targets: torch.Size([64, 27])


In [13]:
from src.model_attention import Encoder, AttentionDecoder, Seq2SeqAttention
import torch.nn as nn
DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

EMBEDDING_DIM = 256
HIDDEN_DIM = 256
NUM_LAYERS = 1
DROPOUT = 0.2
encoder = Encoder(
    input_dim=ru_sp.get_piece_size(),
    embed_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=RU_PAD_IDX,
    dropout=DROPOUT
)

decoder = AttentionDecoder(
    output_dim=en_sp.get_piece_size(),
    embed_dim=EMBEDDING_DIM,
    hidden_dim=HIDDEN_DIM,
    num_layers=NUM_LAYERS,
    pad_idx=EN_PAD_IDX,
    attention_dim=HIDDEN_DIM,
    dropout=DROPOUT
)

model = Seq2SeqAttention(encoder, decoder).to(DEVICE)
criterion = nn.CrossEntropyLoss(
    ignore_index=EN_PAD_IDX
)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3
)

In [14]:
sources, source_lengths, targets = next(iter(train_loader))

with torch.no_grad():
    outputs = model(
        sources.to(DEVICE),
        source_lengths,
        targets.to(DEVICE)
    )

print(outputs.shape)

torch.Size([64, 27, 8000])


In [15]:
from src.train import train_epoch, validate_epoch
import copy
import time

NUM_EPOCHS = 10

train_losses = []
val_losses = []

best_val_loss = float("inf")
best_model_state = None

for epoch in range(1, NUM_EPOCHS + 1):
    start_time = time.time()

    train_loss = train_epoch(
        model,
        train_loader,
        optimizer,
        criterion,
        DEVICE,
        target_pad_idx=EN_PAD_IDX
    )

    val_loss = validate_epoch(
        model,
        valid_loader,
        criterion,
        DEVICE,
        target_pad_idx=EN_PAD_IDX,
        teacher_forcing_rate=0.0
    )

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = copy.deepcopy(model.state_dict())

    epoch_time = time.time() - start_time

    print(
        f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {val_loss:.4f} | "
        f"time: {epoch_time:.1f}s"
    )

Epoch 01/10 | train loss: 2.5148 | val loss: 2.5982 | time: 798.2s
Epoch 02/10 | train loss: 1.8093 | val loss: 2.4773 | time: 807.5s
Epoch 03/10 | train loss: 1.6312 | val loss: 2.4110 | time: 796.7s
Epoch 04/10 | train loss: 1.5324 | val loss: 2.3483 | time: 799.5s
Epoch 05/10 | train loss: 1.4567 | val loss: 2.3179 | time: 797.8s
Epoch 06/10 | train loss: 1.4061 | val loss: 2.3252 | time: 799.5s
Epoch 07/10 | train loss: 1.3610 | val loss: 2.3244 | time: 800.8s
Epoch 08/10 | train loss: 1.3286 | val loss: 2.2891 | time: 796.6s
Epoch 09/10 | train loss: 1.2974 | val loss: 2.2609 | time: 796.4s
Epoch 10/10 | train loss: 1.2741 | val loss: 2.2906 | time: 798.8s


In [ ]:
from pathlib import Path

Path("models").mkdir(exist_ok=True)

checkpoint = {
    "model_state_dict": best_model_state,
    "model_config": {
        "architecture": "bpe_lstm_bahdanau_attention",
        "input_dim": ru_sp.get_piece_size(),
        "output_dim": en_sp.get_piece_size(),
        "embedding_dim": EMBEDDING_DIM,
        "hidden_dim": HIDDEN_DIM,
        "attention_dim": HIDDEN_DIM,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
    },
    "tokenizer_config": {
        "ru_model_path": "artifacts/tokenizers/ru_bpe.model",
        "en_model_path": "artifacts/tokenizers/en_bpe.model",
    },
    "best_val_loss": best_val_loss,
}

torch.save(checkpoint, "models/best_bpe_attention.pt")

In [17]:
import torch
from tqdm.auto import tqdm

model.load_state_dict(best_model_state)
model.to(DEVICE)
model.eval()


def translate_sentence_bpe_attention(
    sentence,
    model,
    ru_sp,
    en_sp,
    device,
    max_length=50
):
    model.eval()

    source_ids = ru_sp.encode(sentence, out_type=int)

    source = torch.tensor(
        [source_ids],
        dtype=torch.long,
        device=device
    )

    source_lengths = torch.tensor(
        [len(source_ids)],
        dtype=torch.long
    )

    predicted_ids = []

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(
            source,
            source_lengths
        )

        source_positions = torch.arange(
            source.size(1),
            device=device
        ).unsqueeze(0)

        source_mask = source_positions < source_lengths.to(
            device
        ).unsqueeze(1)

        input_token = torch.tensor(
            [en_sp.bos_id()],
            dtype=torch.long,
            device=device
        )

        for _ in range(max_length):
            prediction, hidden, cell, _ = model.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                source_mask
            )

            next_token_id = prediction.argmax(dim=1).item()

            if next_token_id == en_sp.eos_id():
                break

            predicted_ids.append(next_token_id)

            input_token = torch.tensor(
                [next_token_id],
                dtype=torch.long,
                device=device
            )

    return en_sp.decode(predicted_ids)

In [18]:
prediction = translate_sentence_bpe_attention(
    row["russian"],
    model,
    ru_sp,
    en_sp,
    DEVICE
)
print("RU:       ", row["russian"])
print("REFERENCE:", row["english"])
print("PREDICT:  ", prediction)
print()

RU:        Вам надо было сразу же мне позвонить.
REFERENCE: You should've phoned me right away.
PREDICT:   You should' phoned me right away.



In [20]:
import sacrebleu

predictions = [
    translate_sentence_bpe_attention(
        russian,
        model,
        ru_sp,
        en_sp,
        DEVICE
    )
    for russian in tqdm(valid["russian"])
]

references = valid["english"].tolist()

bleu = sacrebleu.corpus_bleu(
    predictions,
    [references]
)

chrf = sacrebleu.corpus_chrf(
    predictions,
    [references]
)

print(f"Validation BLEU: {bleu.score:.2f}")
print(f"Validation chrF: {chrf.score:.2f}")

  0%|          | 0/82407 [00:00<?, ?it/s]

Validation BLEU: 49.07
Validation chrF: 64.04
